# 01 - Ingestion
USGS FDSN (gempa Indonesia 2015-2026) + BMKG (gempa terkini).

In [ ]:
# Setup: BASE_DIR path-agnostik (ganti sesuai lokal / Colab)
import os
import sys

BASE_DIR = os.environ.get("BDA_BASE_DIR", "..")  # contoh Colab: "/content/big-data-aol"
RAW_DIR = os.path.join(BASE_DIR, "data", "raw")
os.makedirs(RAW_DIR, exist_ok=True)

sys.path.insert(0, os.path.join(BASE_DIR, "src"))
import ingestion as ing


In [ ]:
import time

MIN_MAG = 4.0
YEARS = range(2015, 2027)


In [ ]:
# Loop tahun -> cek count -> fetch (split kuartal otomatis kalau > 20000) -> simpan raw JSON
ingestion_log = []

for year in YEARS:
    t0 = time.time()
    try:
        features = ing.usgs_fetch_year(year, MIN_MAG)
    except Exception as exc:
        print(f"[GAGAL] USGS {year}: {exc}")
        ingestion_log.append(dict(sumber=f"usgs_{year}", jumlah_record=0,
                                   ukuran_kb=0, waktu_detik=round(time.time() - t0, 2),
                                   status="gagal"))
        continue

    out_path = os.path.join(RAW_DIR, f"usgs_{year}.json")
    ing.save_json({"features": features}, out_path)

    ingestion_log.append(dict(
        sumber=f"usgs_{year}",
        jumlah_record=len(features),
        ukuran_kb=round(os.path.getsize(out_path) / 1024, 1),
        waktu_detik=round(time.time() - t0, 2),
        status="ok",
    ))
    print(f"USGS {year}: {len(features)} event")


In [ ]:
# BMKG gempaterkini + autogempa
t0 = time.time()
bmkg_terkini = ing.bmkg_fetch_gempaterkini()
bmkg_auto = ing.bmkg_fetch_autogempa()

bmkg_out = {"gempaterkini": bmkg_terkini, "autogempa": bmkg_auto}
bmkg_path = os.path.join(RAW_DIR, "bmkg_terkini.json")
ing.save_json(bmkg_out, bmkg_path)

n_bmkg = len(bmkg_terkini.get("Infogempa", {}).get("gempa", []))
ingestion_log.append(dict(
    sumber="bmkg_terkini",
    jumlah_record=n_bmkg,
    ukuran_kb=round(os.path.getsize(bmkg_path) / 1024, 1),
    waktu_detik=round(time.time() - t0, 2),
    status="ok",
))
print(f"BMKG: {n_bmkg} event terkini")


In [ ]:
import pandas as pd

log_df = pd.DataFrame(ingestion_log)
log_df.to_csv(os.path.join(RAW_DIR, "_ingestion_log.csv"), index=False)
log_df
